## Dataset:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, random_split
import matplotlib as plt
import os
from PIL import Image
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)


dataset = datasets.ImageFolder('/content/gdrive/My Drive/Sports10', transform=None)


def is_corrupt(path):
    try:
        img = Image.open(path)
        img.verify()
        return False
    except:
        return True

# Scan and delete corrupt images
def clean_dataset(folder_path):
    removed = 0
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp')):
                path = os.path.join(root, file)
                if is_corrupt(path):
                    os.remove(path)
                    removed += 1
    print(f"✅ Corrupt image cleanup complete. Removed {removed} images.")


Mounted at /content/gdrive


In [ ]:
clean_dataset('/content/gdrive/My Drive/Sports10')

In [ ]:
train_size = int(0.8 * len(dataset))  # 80% for training
val_size = len(dataset) - train_size  # 20% for validation

train_dataset, test_dataset = random_split(dataset, [train_size, val_size])

train_dataset.dataset.transform =train_transforms
test_dataset.dataset.transform = test_transforms


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### Data Preprocessing:
1. Remove corrupted or low-quality images.

2. Standardize image sizes (commonly 224x224 for transfer learning models like ResNet or VGG).

### Exploratory Data Analysis (EDA)
1. Visualize class distribution (are some sports underrepresented?).

2. Check for lighting, resolution, and visual style diversity.

### Model selection:
1. Replace the final layer with nn.Linear matching your 10 sports genres.

2. Fine-tune the model: start by freezing most layers, only training the last FC layer, then unfreeze more layers if needed.

## Model:

In [ ]:
model1 = models.resnet18(pretrained=True)
model1
model2 = models.vgg16(pretrained=True)
model2

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 103MB/s]
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed i

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
model2.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [ ]:
for param in model2.parameters():
    param.requires_grad = False

num_features = model2.classifier[6].in_features
model2.classifier[6] = nn.Linear(num_features, len(dataset.classes))
model2.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1, bias=True)
)

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.classifier.parameters(), lr=0.001)

# Lists to store results
epochs_list=[]
epoch_train_loss = []  # Stores (epoch, training loss)
epoch_test_loss = []   # Stores (epoch, test loss)
epoch_test_acc = []    # Stores (epoch, test accuracy)

# Training parameters
epochs = 5

running_loss = 0
print_every = 5

# Training loop
for epoch in range(epochs):
    for inputs, labels in train_loader:
        model2.train()
        optimizer.zero_grad()

        logps = model2.forward(inputs)
        loss = criterion(logps, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_loss = running_loss / len(train_loader.dataset)
        epoch_train_loss.append(train_loss)
        model2.eval()
        test_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
                for inputs, labels in test_loader:
                    outputs = model2.forward(inputs)
                    loss = criterion(outputs, labels)
                    test_loss += loss.item()

                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

        accuracy = correct / total

        # Store data
        epoch =epochs_list.append(epoch+1)
        epoch_test_loss.append(test_loss / len(test_loader))
        epoch_test_acc.append(accuracy)

        print(f"Epoch {epoch+1}/{epochs}.. "
              f"Train Loss: {running_loss/print_every:.3f}.. "
              f"Test Loss: {test_loss/len(test_loader.dataset):.3f}.. "
              f"Test Accuracy: {accuracy:.3f}")

In [ ]:

plt.figure(figsize=(12, 6))

# Plot Train Loss
plt.subplot(1, 3, 1)  # 1 row, 3 columns, first plot
plt.plot(epochs_list, epoch_train_loss, label="Train Loss", color='blue')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

# Plot Test Loss
plt.subplot(1, 3, 2)  # 1 row, 3 columns, second plot
plt.plot(epochs_list, epoch_test_loss, label="Test Loss", color='red')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Test Loss')
plt.grid(True)

# Plot Test Accuracy
plt.subplot(1, 3, 3)  # 1 row, 3 columns, third plot
plt.plot(epochs_list, epoch_test_acc, label="Test Accuracy", color='green')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Test Accuracy')
plt.grid(True)

# Show the plots
plt.tight_layout()
plt.show()

## Custom CNN

In [ ]:
def __init__(self, num_classes=10):
    super(SportsNet, self).__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=5, padding=2),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(128*24*24, 512),
        nn.ReLU(),
        nn.Linear(512, num_classes)
    )

def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


model = SportsNet(num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10


train_losses = []
test_losses = []
test_accuracies = []


for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()


    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    model.eval()
    test_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()

    epoch_test_loss = test_loss / len(test_loader)
    epoch_accuracy = correct / len(test_loader.dataset)

    test_losses.append(epoch_test_loss)
    test_accuracies.append(epoch_accuracy)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Test Loss: {epoch_test_loss:.4f} | "
          f"Accuracy: {epoch_accuracy*100:.2f}%")

In [ ]:

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(range(1, epochs+1), train_losses, 'b-o')
plt.title("Training Loss")
plt.xlabel("Epochs")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(range(1, epochs+1), test_losses, 'r-o')
plt.title("Test Loss")
plt.xlabel("Epochs")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(range(1, epochs+1), test_accuracies, 'g-o')
plt.title("Test Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.grid(True)

plt.tight_layout()
plt.show()